# Content-Based Book Recommender with ChromaDB

We build a content-based recommender over ~1,200 book summaries. Instead of materialising a
full book-to-book similarity matrix with `pairwise_distances`, we store the summary embeddings
in a **ChromaDB** vector store and let it do the nearest-neighbour search for us.

Two retrieval paths are supported:

1. **`get_similar_books(title)`** - "more books like *this* book". The book's own embedding is
   pulled from the store and used as the query vector.
2. **`search_books(query)`** - "find me books about *this* idea". A free-text keyword or phrase
   typed by the user is embedded on the fly and used as the query vector.

Both paths hit the same collection. That is the whole point of a vector store: a query is just
another vector, whether it came from a document or from a user.

*Manaranjan Pradhan / AIThinking.in*

## Setup

In [ ]:
# !pip install -q chromadb sentence-transformers

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Loading the Book Dataset

In [ ]:
import pandas as pd
import numpy as np
import ast

In [ ]:
# Set the maximum column width to 100 characters
pd.set_option('display.max_colwidth', 500)

In [ ]:
books_df = pd.read_csv('books.csv', index_col=[0])

In [ ]:
books_df.head(10)

In [ ]:
books_df.info()

In [ ]:
books_df.shape

In [ ]:
books_df.categories.value_counts()

## Vectorizing the Summaries: the TF-IDF Baseline

Before we bring in ChromaDB, it is worth keeping the classical vectorizer around, because it
makes the limitation we are trying to escape very concrete.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Create tf-idf vectors using the overview column
tfidf_vectorizer = TfidfVectorizer(stop_words='english',
                                   min_df = 5,
                                   max_df = 0.6)

In [ ]:
tfidf_matrix = tfidf_vectorizer.fit_transform(books_df['summaries'])

In [ ]:
tfidf_matrix.shape

In [ ]:
tfidf_df = pd.DataFrame(tfidf_matrix.toarray())

In [ ]:
vocabs = list(tfidf_vectorizer.vocabulary_.keys())

In [ ]:
tfidf_df.columns = vocabs

In [ ]:
from random import sample

sample_vocabs = sample(vocabs,10)

In [ ]:
tfidf_df.sample(10)[sample_vocabs]

### Why we do not stop here

The TF-IDF matrix has two problems for our purpose.

**It is vocabulary-bound.** With `min_df=5, max_df=0.6` the vocabulary collapses to ~900 terms.
A user who types *"how do I stop procrastinating"* only gets a meaningful vector if those exact
words survived the pruning. Anything else becomes a zero vector, and a zero vector is equidistant
from everything.

**It has no notion of meaning.** *"money"* and *"wealth"* are two unrelated columns. Two summaries
that say the same thing in different words look dissimilar.

Dense embeddings from a sentence transformer fix both. And once we have embeddings, we no longer
want to compute a 1227 x 1227 similarity matrix by hand - that is O(n^2) memory, has to be rebuilt
whenever a book is added, and cannot answer a query that is not already a row in the matrix.
A vector store handles indexing, search, filtering and persistence for us.

(The TF-IDF vectors are not wasted - we put them into Chroma in the appendix at the bottom and
compare the two side by side.)

## Setting up the ChromaDB Vector Store

Three moving parts:

- **Embedding function** - `all-MiniLM-L6-v2`, a 384-dimensional sentence transformer. Chroma calls
  it for us on both the documents we add and the queries we run, so the two always land in the same
  space.
- **Client** - `PersistentClient` writes to disk, so the embeddings survive a kernel restart. Use
  `chromadb.EphemeralClient()` if you want a purely in-memory store.
- **Collection** - the table. We set the distance metric to cosine, which is what we were computing
  by hand earlier.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

In [ ]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name = "all-MiniLM-L6-v2"
)

In [ ]:
chroma_client = chromadb.PersistentClient(path = "./chroma_books_db")

In [ ]:
book_collection = chroma_client.get_or_create_collection(
    name = "books",
    embedding_function = embedding_fn,
    metadata = {"hnsw:space": "cosine"}
)

### Loading the summaries into the collection

Every record needs three things:

- `ids` - our key back into `books_df`. We use the dataframe index as a string.
- `documents` - the raw summary text. Chroma embeds it for us.
- `metadatas` - anything we want to filter on or display later, here the book name and category.

We add in batches and skip the work if the collection is already populated, so re-running the
notebook does not re-embed 1,227 summaries.

In [ ]:
def load_books_into_chroma(df, collection, batch_size = 200):
    if collection.count() == len(df):
        print(f"Collection already has {collection.count()} books. Skipping load.")
        return

    ids       = [str(idx) for idx in df.index]
    documents = df['summaries'].tolist()
    metadatas = [{"book_name": row.book_name, "categories": row.categories}
                 for row in df.itertuples()]

    for start in range(0, len(df), batch_size):
        end = start + batch_size
        collection.add(ids       = ids[start:end],
                       documents = documents[start:end],
                       metadatas = metadatas[start:end])
        print(f"Added books {start} to {min(end, len(df)) - 1}")

    print(f"Done. Collection size: {collection.count()}")

In [ ]:
load_books_into_chroma(books_df, book_collection)

In [ ]:
book_collection.count()

### What does a stored record look like?

`peek()` returns the first few records. Note the embedding: 384 dense floats, in contrast to the
900 mostly-zero TF-IDF columns.

In [ ]:
sample_record = book_collection.peek(limit = 2)

print("Keys stored     :", [k for k, v in sample_record.items() if v is not None])
print("IDs             :", sample_record['ids'])
print("Metadata        :", sample_record['metadatas'])
print("Embedding shape :", sample_record['embeddings'].shape)
print("First 8 dims    :", sample_record['embeddings'][0][:8].round(3))

## A Helper to Format Chroma's Results

Chroma returns a dict of parallel lists, one entry per query. Since we always send a single query,
we take element `[0]` of each list and flatten it into a dataframe.

Chroma reports **cosine distance**, so we convert it back to the similarity score we are used to:

$$\text{similarity} = 1 - \text{cosine distance}$$

In [ ]:
def to_dataframe(results):
    records = []

    for book_id, meta, doc, dist in zip(results['ids'][0],
                                        results['metadatas'][0],
                                        results['documents'][0],
                                        results['distances'][0]):
        records.append({
            "book_id"    : int(book_id),
            "book_name"  : meta['book_name'],
            "categories" : meta['categories'],
            "summaries"  : doc,
            "similarity" : round(1 - dist, 4)
        })

    return pd.DataFrame(records)

## Path 1: Finding TopN Similar Books (book as the query)

Same idea as before, but no similarity matrix. We look up the book's stored embedding with `get()`
and hand that vector straight back to `query()`.

We ask for `topN + 1` neighbours because the closest match to a book is always itself, and then
drop it. The optional `category` argument shows off Chroma's metadata filter - the search runs only
over the books that match, instead of ranking everything and filtering afterwards.

In [ ]:
def get_similar_books(title, topN = 5, category = None):
    # Resolve the title: exact match first, then a case-insensitive partial match
    matches = books_df[books_df['book_name'] == title]

    if len(matches) == 0:
        matches = books_df[books_df['book_name'].str.contains(title, case = False, regex = False)]

    if len(matches) == 0:
        raise ValueError(f"No book found matching '{title}'")

    book_idx  = matches.index[0]
    book_name = books_df.loc[book_idx, 'book_name']

    # Pull the embedding Chroma already computed for this book
    stored = book_collection.get(ids = [str(book_idx)], include = ["embeddings"])
    book_embedding = stored['embeddings']

    where_filter = {"categories": category} if category else None

    results = book_collection.query(
        query_embeddings = book_embedding,
        n_results        = topN + 1,
        where            = where_filter,
        include          = ["documents", "metadatas", "distances"]
    )

    similar_df = to_dataframe(results)

    # The book itself is always the nearest neighbour. Drop it.
    similar_df = similar_df[similar_df.book_id != book_idx].head(topN)

    print(f"Books similar to: {book_name}")
    return similar_df.reset_index(drop = True)

### Finding Similar Books

 - The Miracle of Mindfulness
 - The Bitcoin Standard
 - Mind Gym
 - Get A Financial Life

In [ ]:
books_df[books_df.book_name.str.contains("Mindfulness")]

In [ ]:
get_similar_books('The Miracle of Mindfulness')

In [ ]:
get_similar_books('The Bitcoin Standard')

In [ ]:
get_similar_books('Mind Gym')

In [ ]:
get_similar_books('Get A Financial Life')

### Recommending within a category

Passing `category` pushes the constraint down into the vector store, so the nearest-neighbour search
is only performed over books in that category.

In [ ]:
get_similar_books('The Bitcoin Standard', topN = 5, category = 'economics')

## Path 2: Searching Books by Keyword or Phrase (text as the query)

This is what the similarity matrix could never do. The user types anything - a keyword, a phrase, a
whole sentence describing what they are in the mood for - and Chroma embeds it with the *same*
model, then finds the summaries closest to it.

Note that nothing about the collection changes. We only swap `query_embeddings` for `query_texts`.
There is no keyword matching happening here: a query can retrieve a book whose summary does not
contain a single word of the query, as long as the meaning lines up.

In [ ]:
def search_books(query, topN = 5, category = None):
    where_filter = {"categories": category} if category else None

    results = book_collection.query(
        query_texts = [query],
        n_results   = topN,
        where       = where_filter,
        include     = ["documents", "metadatas", "distances"]
    )

    print(f"Search results for: '{query}'" + (f" (category: {category})" if category else ""))
    return to_dataframe(results)

### Single keyword

In [ ]:
search_books("procrastination")

### A phrase

In [ ]:
search_books("how to build good habits and break bad ones")

### A full sentence describing an intent

Watch the top results here. The user never says *"negotiation"* or *"persuasion"*, but the retrieved
books are about exactly that. This is the payoff of semantic embeddings over TF-IDF's exact-token
matching.

In [ ]:
search_books("I want to become better at convincing people in difficult conversations")

### Search restricted to a category

In [ ]:
search_books("saving and investing for retirement", topN = 5, category = "money")

### A query with no good answer in the catalogue

Vector search always returns *something* - the nearest neighbours exist no matter how far away they
are. The similarity scores are the tell: they drop sharply when nothing in the catalogue is really
about the query. In production you would set a similarity floor (say, 0.35) and return an empty
result below it.

In [ ]:
search_books("how to bake sourdough bread at home")

In [ ]:
def search_books_with_threshold(query, topN = 5, min_similarity = 0.35):
    results_df = search_books(query, topN = topN)
    results_df = results_df[results_df.similarity >= min_similarity]

    if len(results_df) == 0:
        print("No book in the catalogue is close enough to this query.")

    return results_df

In [ ]:
search_books_with_threshold("how to bake sourdough bread at home")

In [ ]:
search_books_with_threshold("dealing with anxiety and negative thoughts")

## Appendix: Putting the TF-IDF Vectors into Chroma Instead

Chroma does not force you to use its embedding function. You can pass your own vectors with
`embeddings=` and it becomes a pure index over whatever representation you already have. Doing this
with the TF-IDF matrix gives us a clean, controlled comparison against the sentence-transformer
collection - the store is identical, only the vectors differ.

This is a useful cell to run in class: it shows the recommendation quality holding up reasonably for
book-to-book similarity, and then falling apart for free-text queries.

In [ ]:
tfidf_collection = chroma_client.get_or_create_collection(
    name       = "books_tfidf",
    metadata   = {"hnsw:space": "cosine"}
)

if tfidf_collection.count() != len(books_df):
    tfidf_collection.add(
        ids        = [str(idx) for idx in books_df.index],
        documents  = books_df['summaries'].tolist(),
        embeddings = tfidf_matrix.toarray().tolist(),        # our own 900-dim sparse vectors
        metadatas  = [{"book_name": r.book_name, "categories": r.categories}
                      for r in books_df.itertuples()]
    )

print("TF-IDF collection size:", tfidf_collection.count())

In [ ]:
def search_books_tfidf(query, topN = 5):
    # The query must be projected into the SAME vocabulary space
    query_vector = tfidf_vectorizer.transform([query]).toarray()

    if query_vector.sum() == 0:
        print(f"'{query}' -> zero vector. None of these words are in the 900-term vocabulary.")

    results = tfidf_collection.query(
        query_embeddings = query_vector.tolist(),
        n_results        = topN,
        include          = ["documents", "metadatas", "distances"]
    )

    return to_dataframe(results)

Ask the TF-IDF collection the same intent-style question. The vocabulary was pruned to ~900 terms,
so most of the query is discarded before it is even scored - and the results reflect that.

In [ ]:
search_books_tfidf("I want to become better at convincing people in difficult conversations")

In [ ]:
search_books("I want to become better at convincing people in difficult conversations")

### Housekeeping

Chroma persisted everything to `./chroma_books_db`. Uncomment below to wipe the collections and
start over.

In [ ]:
# chroma_client.delete_collection("books")
# chroma_client.delete_collection("books_tfidf")
# chroma_client.list_collections()